In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Load relevant packages

In [2]:
import os
import math
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import json, time, pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import gc, time, json
from sklearn.metrics import precision_score
import pandas as pd

# Define Simon's origonal SeqRR model based on Ridge Regression

In [3]:
class SeqRR(nn.Module):
    def __init__(self, embedding_dim, dropout_rate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.linear = nn.Linear(embedding_dim, 1)
    def forward(self, emb):
        x = self.dropout(emb)
        logits = self.linear(x)
        return logits.squeeze(1)

# Define EpiNN models with (AAIndex-based) first and second order effect considerations

In [4]:
class EpiNN(nn.Module):
    def __init__(self, L, D, dropout_rate=0.0):
        super().__init__()
        self.L = L
        self.D = D
        self.dropout = nn.Dropout(dropout_rate)
        self.token_linear = nn.Linear(L*D+1, 1)
        self.interaction_mlp = nn.Sequential(
            nn.Linear(2*D, D),
            nn.LeakyReLU(),
            nn.Linear(D, 16),
            nn.LeakyReLU(),
            nn.Linear(16, 1)
        )
        self.interaction_scale = nn.Parameter(torch.tensor(0.2))

    def forward(self, emb):
        # compute first order interactions
        x1 = self.dropout(emb)
        x1 = self.token_linear(x1).squeeze(-1)

        # compute second order interactions
        x2 = emb[:,:-1].reshape(emb.shape[0],self.L, self.D)
        mask = (x2.abs().sum(dim=-1) > 0).float()  # (N, L)
        mask = mask[:, :, None] * mask[:, None, :] # (N, L, L)
        w = self.token_linear.weight.squeeze(0)[:-1].view(1, self.L, self.D)
        x2 = x2 * w
        xi = x2.unsqueeze(2).expand(-1, -1, self.L, -1)  # (N, L, L, D) copies x[:, i, :]
        xj = x2.unsqueeze(1).expand(-1, self.L, -1, -1)  # (N, L, L, D) copies x[:, j, :]
        x2 = torch.cat([(xi+xj)/2, (xi-xj).abs()], dim=-1)          # (N, L, L, 2D)
        x2 = self.interaction_mlp(x2).squeeze(-1) * mask   # (N, L, L)
        x2 = torch.triu(x2, diagonal=1).sum(dim=(1, 2)) * self.interaction_scale
        return x1 + x2

# Define EpiATT model with (ESM+attention-based) first and second order effect considerations

In [5]:
# EpiNN model using low-rank attention to calculate epistasis
class EpiATT(nn.Module):


    def __init__(self, L, D, D_hidden=64, n_heads=4, dropout_rate=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.D_hidden = D_hidden or D
        assert D_hidden % n_heads == 0, "D_hidden must be divisible by n_heads"
        self.head_dim = D_hidden // n_heads

        # First-order additive components
        self.token_linear = nn.Linear(D, 1, bias=False)
        self.seq_linear   = nn.Linear(L, 1, bias=True)

        # Multi-Head Low-Rank Projection
        self.projection = nn.Linear(D, D_hidden, bias=False)

        self.dropout = nn.Dropout(dropout_rate)
        self.interaction_scale = nn.Parameter(torch.tensor(1.0))


    def forward(self, x, esm_priors=None):
        N, L, _ = x.shape

        # --- First-order fitness effect calculation ---
        first_order = self.token_linear(x).squeeze(-1)       # (N, L)
        first_order = self.seq_linear(first_order).squeeze(-1)  # (N,)

        # --- second-order fitness effect calculation ---
        h = self.projection(self.dropout(x))
        h = h.view(N, L, self.n_heads, self.head_dim).transpose(1, 2) # (N, H, L, Dh)
        attn_scores = torch.matmul(h, h.transpose(-1, -2)) / math.sqrt(self.head_dim) # (N, H, L, L)
        attn_scores = attn_scores.mean(dim=1) # Average over heads: (N, L, L)

        if esm_priors is not None:
            attn_scores = attn_scores * esm_priors

        second_order = torch.triu(attn_scores, diagonal=1).sum(dim=(1, 2)) * self.interaction_scale

        return first_order + second_order

# Decision-tree like ESM based model

In [6]:
class SeqDT(nn.Module):
    def __init__(self, L, D, dropout_rate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.token_linear = nn.Linear(D, 1, bias=True)
        self.res_weight = nn.Parameter(torch.ones(1, L) * 0.1, requires_grad=True)
        self.mlp = nn.Sequential(
            nn.Linear(D, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.dropout(x)
        mask = (x.abs().sum(dim=-1) > 0).float()
        w = self.token_linear(x).squeeze(-1) + self.res_weight  # (N, L)
        w = w * mask
        w = w.unsqueeze(-1)          # (N, L, 1)
        x = (x * w).sum(dim=1)    # (N, D)  weighted sum
        x = self.mlp(x).squeeze(-1)
        return x

# Define helper functions for training

In [7]:
CENSOR_THRESH = -5.0
LAMBDA_NEG = 0.5  # weight for negatives


def load_data_single(emb, attn, labels, batch, batch_size, device):
    """
    Loads a batch of data from the single-embedding dataset.

    Returns:
        data_batch: Tensor [B, embedding_dim]
        labels_batch: Tensor [B]
    """
    if (batch+1)*batch_size > len(labels):
        idx = np.arange(batch*batch_size, len(labels))
    else:
        idx = np.arange(batch*batch_size, (batch+1)*batch_size)

    data_batch = emb[idx]
    if attn is not None:
        attn_batch = attn[idx]
    else:
        attn_batch = None
    labels_batch = labels[idx]

    # --- Shape guards (edge cases with single element) ---
    if data_batch.ndim == 1:          # -> [D]  -> make [1, D]
        data_batch = np.expand_dims(data_batch, axis=0)
    if np.isscalar(labels_batch):      # -> scalar -> make [1]
        labels_batch = np.array([labels_batch], dtype=np.float32)
    elif labels_batch.ndim == 0:       # -> [] -> make [1]
        labels_batch = labels_batch.reshape(1)

    data_batch = torch.from_numpy(data_batch).to(torch.float32).to(device)
    if attn_batch is not None:
        attn_batch = torch.from_numpy(attn_batch).to(torch.float32).to(device)
    labels_batch = torch.from_numpy(labels_batch).to(torch.float32).to(device)

    return data_batch, attn_batch, labels_batch


def pairwise_rank_loss_pos(preds_pos, targets_pos, margin=0.0, min_delta=0.0, reduction="mean"):
    """
    Pairwise logistic ranking loss over positive samples only.
    Encourages: if target_i > target_j then pred_i > pred_j by (margin).
    min_delta: ignore pairs with |target_i - target_j| <= min_delta (reduces noise).
    """
    # preds_pos, targets_pos: (P,)
    if preds_pos.numel() < 2:
        return preds_pos.new_tensor(0.0)

    # Pairwise differences
    dy = targets_pos[:, None] - targets_pos[None, :]   # (P, P)
    dp = preds_pos[:, None] - preds_pos[None, :]       # (P, P)

    # Mask pairs where target_i > target_j (and optionally sufficiently separated)
    mask = dy > min_delta
    if not mask.any():
        return preds_pos.new_tensor(0.0)

    # Logistic ranking loss: softplus(-(dp - margin)) for desired dp >= margin
    loss_mat = F.softplus(-(dp - margin))

    loss = loss_mat[mask]
    if reduction == "sum":
        return loss.sum()
    return loss.mean()


def censored_huber_plus_rank_loss(
    model,
    preds,
    targets,
    threshold=CENSOR_THRESH,
    lambda_neg=LAMBDA_NEG,
    reduction="mean",
    l1_lambda=0.0,
    l2_lambda=0.0,
    huber_delta=1.0,          # SmoothL1 beta
    rank_lambda=1.0,         # weight for rank loss
    rank_margin=0.0,
    rank_min_delta=0.0,       # ignore tiny target diffs among positives
):
    """
    Positives (targets > threshold): Huber loss on (pred-target)
    Negatives (targets <= threshold): censored penalty on max(pred-threshold, 0)
    + optional pairwise rank loss among positives
    + optional L1/L2 reg on model parameters
    """
    preds = preds.squeeze(-1)
    targets = targets.squeeze(-1)

    pos = targets > threshold
    neg = ~pos

    loss_vec = torch.zeros_like(targets)

    if pos.any():
        loss_pos = F.smooth_l1_loss(
            preds[pos], targets[pos],
            reduction="none",
            beta=huber_delta
        )
        loss_vec[pos] = loss_pos

    # --- Negative: censored penalty ---
    if neg.any():
        over = F.relu(preds[neg] - threshold)
        loss_vec[neg] = lambda_neg * F.smooth_l1_loss(over, torch.zeros_like(over), reduction="none", beta=huber_delta)

    # --- Aggregate data loss ---
    if reduction == "sum":
        data_loss = loss_vec.sum()
    else:
        data_loss = loss_vec.mean()

    # --- Pairwise rank loss (positives only) ---
    if rank_lambda > 0.0 and pos.any():
        rank_loss = pairwise_rank_loss_pos(
            preds[pos], targets[pos],
            margin=rank_margin,
            min_delta=rank_min_delta,
            reduction="mean"
        )
        data_loss = data_loss + rank_lambda * rank_loss

    # --- Regularization ---
    if l1_lambda == 0.0 and l2_lambda == 0.0:
        return data_loss

    reg_l1 = sum(p.abs().sum() for p in model.parameters())
    reg_l2 = sum(p.pow(2).sum() for p in model.parameters())
    reg = l1_lambda * reg_l1 + l2_lambda * reg_l2

    return data_loss + reg



def censored_mse_loss(model, preds, targets, threshold=CENSOR_THRESH, lambda_neg=LAMBDA_NEG, reduction="mean", l1_lambda=0.0, l2_lambda=0.0):
    preds = preds.squeeze(-1)
    targets = targets.squeeze(-1)

    pos = targets > threshold
    neg = ~pos

    loss_vec = torch.zeros_like(targets)

    if pos.any():
        diff_pos = preds[pos] - targets[pos]
        loss_vec[pos] = diff_pos.pow(2)

    if neg.any():
        over = F.relu(preds[neg] - threshold)
        loss_vec[neg] = lambda_neg * over.pow(2)

    if reduction == "mean":
        data_loss = loss_vec.mean()
    elif reduction == "sum":
        data_loss = loss_vec.sum()

    # add the L1 and L2 regularization terms
    if l1_lambda == 0.0 and l2_lambda == 0.0:
        return data_loss

    reg_l1 = sum(p.abs().sum() for p in model.parameters())
    reg_l2 = sum(p.pow(2).sum() for p in model.parameters())
    reg = l1_lambda * reg_l1 + l2_lambda * reg_l2

    return data_loss + reg



def train_epoch_single(model, optimizer, emb, attn, labels, batch_size, l1_lambda, l2_lambda, loss, device):  # loss: 'MSE' or 'Huber'
    model.train()
    running_loss = 0.0
    total_samples = 0
    num_batches = math.ceil(len(labels) / batch_size)

    for batch in range(num_batches):
        data_batch, attn_batch, labels_batch = load_data_single(emb, attn, labels, batch, batch_size, device)
        if attn_batch is not None:
            preds = model(data_batch, attn_batch).squeeze(-1)
        else:
            preds = model(data_batch).squeeze(-1)
        if loss == 'MSE':
            loss = censored_mse_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        else:
            loss = censored_huber_plus_rank_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = data_batch.size(0)
        running_loss += loss.item() * bs
        total_samples += bs

    avg_loss = running_loss / total_samples
    return avg_loss


def test_epoch_single(model, emb, attn, labels, batch_size, l1_lambda, l2_lambda, loss, device): # loss: 'MSE' or 'Huber'
    model.eval()
    running_loss = 0.0
    total_samples = 0
    all_preds, all_targets = [], []
    num_batches = math.ceil(len(labels) / batch_size)

    with torch.no_grad():
        for batch in range(num_batches):
            data_batch, attn_batch, labels_batch = load_data_single(emb, attn, labels, batch, batch_size, device)

            # Skip truly empty batch (can happen if len(labels) == 0)
            if data_batch.numel() == 0:
                continue
            if attn_batch is not None:
                preds = model(data_batch, attn_batch)            # [B, 1] or [B]
            else:
                preds = model(data_batch)            # [B, 1] or [B]
            preds = preds.squeeze(-1).reshape(-1)        # force [B]
            labels_batch = labels_batch.reshape(-1)      # force [B]

            if loss == 'MSE':
                loss = censored_mse_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
            else:
                loss = censored_huber_plus_rank_loss(model, preds, labels_batch, l1_lambda=l1_lambda, l2_lambda=l2_lambda)

            bs = preds.size(0)
            running_loss += loss.item() * bs
            total_samples += bs
            all_preds.append(preds)
            all_targets.append(labels_batch)

    if total_samples == 0:
        # No samples: return NaNs and empty tensors
        return float('nan'), float('nan'), float('nan'), torch.tensor([]), torch.tensor([])

    avg_loss = running_loss / total_samples
    # Filter out any accidental empties and cat safely
    all_preds = torch.cat([t.reshape(-1) for t in all_preds if t.numel() > 0], dim=0)
    all_targets = torch.cat([t.reshape(-1) for t in all_targets if t.numel() > 0], dim=0)

    # Spearman on positives only
    pos_mask = all_targets > CENSOR_THRESH
    if pos_mask.sum().item() >= 2:
        spearman_corr, _ = spearmanr(
            all_preds[pos_mask].cpu().numpy(),
            all_targets[pos_mask].cpu().numpy()
        )
    else:
        spearman_corr = np.nan

    # Accuracy on negatives
    neg_mask = ~pos_mask
    if neg_mask.any():
        neg_accuracy = (all_preds[neg_mask] <= CENSOR_THRESH).float().mean().item()
    else:
        neg_accuracy = np.nan

    return avg_loss, spearman_corr, neg_accuracy, all_preds, all_targets

# Define helper functions for plots & eval

In [8]:
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def plot_training_curves(history, title_prefix, outdir, fold):
    """
    history: dict with keys 'train_loss', 'val_loss', 'spearman', 'neg_acc'
    Curves start at epoch 0 (untrained / zero-skill baseline).
    """
    ensure_dir(outdir)
    epochs = np.arange(len(history['train_loss']))  # 0..E

    fig = plt.figure(figsize=(10, 8))
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(epochs, history['train_loss'], label='Train Loss')
    ax1.set_title('Train Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

    ax2 = fig.add_subplot(2, 2, 2)
    ax2.plot(epochs, history['val_loss'], label='Val Loss')
    ax2.set_title('Validation Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')

    ax3 = fig.add_subplot(2, 2, 3)
    ax3.plot(epochs, history['spearman'], label='Spearman (pos)')
    ax3.set_title('Spearman (positives)')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Spearman ρ')

    ax4 = fig.add_subplot(2, 2, 4)
    ax4.plot(epochs, history['neg_acc'], label='Accuracy (neg)')
    ax4.set_title('Accuracy (negatives)')
    ax4.set_xlabel('Epoch'); ax4.set_ylabel('Accuracy')

    plt.suptitle(f'{title_prefix} — Fold {fold}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    out_path = os.path.join(outdir, f'{title_prefix.replace(" ", "_").lower()}_fold{fold}_curves.png')
    plt.savefig(out_path, dpi=150)
    plt.close(fig)

def aggregate_and_plot(all_preds, all_targets, dataset_name, outdir):
    """
    all_preds, all_targets: list of torch tensors per fold
    """
    ensure_dir(outdir)
    preds = torch.cat(all_preds).cpu().numpy()
    targets = torch.cat(all_targets).cpu().numpy()

    # 1) Scatter with Spearman
    pos_mask = targets > CENSOR_THRESH
    sp, _ = spearmanr(preds[pos_mask], targets[pos_mask])
    neg_acc = np.mean(preds[~pos_mask] <= CENSOR_THRESH)
    fig = plt.figure(figsize=(6, 6))
    plt.scatter(targets, preds, s=10, alpha=0.6)
    plt.xlabel('True'); plt.ylabel('Predicted')
    plt.title(f'{dataset_name} — Pred vs True (Spearman ρ={sp:.3f})')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_scatter.png'), dpi=150)
    plt.close(fig)

    # Binary labels for ROC/PR using WT fitness as threshold
    y_true = (targets > 0.0).astype(np.int32)
    y_score = preds  # continuous scores

    # 2) ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    fig = plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f'AUC={roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'{dataset_name} — ROC (AUC={roc_auc:.3f})')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_roc.png'), dpi=150)
    plt.close(fig)

    # 3) PR
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)  # AUPRC
    fig = plt.figure(figsize=(6, 6))
    plt.plot(recall, precision, label=f'AUPRC={ap:.3f}')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'{dataset_name} — PR (AUPRC={ap:.3f})')
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_pr.png'), dpi=150)
    plt.close(fig)

    np.save(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_preds.npy'), preds)
    np.save(os.path.join(outdir, f'{dataset_name.replace(" ", "_").lower()}_targets.npy'), targets)

    return {
        'spearman': float(sp),
        'neg_acc': float(neg_acc),
        'roc_auc': float(roc_auc),
        'auprc': float(ap),
        'preds': preds,
        'targets': targets
    }

In [9]:
def train_on_dataset(model, train_emb, train_attn, train_labels,
                     test_emb, test_attn, test_labels,
                     n_epoch, batch_size, loss, device,
                     title_prefix, outdir, fold,
                     lr=1e-3, weight_decay=1e-2, dropout_rate=0.1,
                     l1_lambda=0.0, l2_lambda=0.0,
                     patience=16):
    """
    Trains a fresh model with early stopping on val loss.
    Adds 'epoch 0' zero-skill metrics to history.
    """

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train_loss': [], 'val_loss': [], 'spearman': [], 'neg_acc': []}

    # ---- Epoch 0: zero-skill/untrained baseline ----
    train0_loss, _, _, _, _ = test_epoch_single(model, train_emb, train_attn, train_labels, batch_size, l1_lambda, l2_lambda, loss, device)
    val0_loss, sp0, neg0, _, _ = test_epoch_single(model, test_emb, test_attn, test_labels, batch_size, l1_lambda, l2_lambda, loss, device)
    history['train_loss'].append(train0_loss)
    history['val_loss'].append(val0_loss)
    history['spearman'].append(sp0 if sp0 is not None else np.nan)
    history['neg_acc'].append(neg0 if neg0 is not None else np.nan)

    best_val = val0_loss
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    epochs_since_improve = 0

    pbar = tqdm(range(1, n_epoch+1), desc=f'{title_prefix} — Fold {fold}', leave=False)

    for epoch in pbar:
        train_loss = train_epoch_single(model, optimizer, train_emb, train_attn, train_labels, batch_size, l1_lambda, l2_lambda, loss, device)
        val_loss, sp, neg_acc, _, _ = test_epoch_single(model, test_emb, test_attn, test_labels, batch_size, l1_lambda, l2_lambda, loss, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['spearman'].append(sp if sp is not None else np.nan)
        history['neg_acc'].append(neg_acc if neg_acc is not None else np.nan)

        pbar.set_postfix({
            'train': f'{train_loss:.4f}',
            'val': f'{val_loss:.4f}',
            'ρ(pos)': f'{(sp if not np.isnan(sp) else 0):.3f}',
            'acc(neg)': f'{(neg_acc if not np.isnan(neg_acc) else 0):.3f}',
            'pat': f'{epochs_since_improve}/{patience}'
        })

        # Early stopping on best val loss
        if val_loss < best_val - 1e-8:  # tiny margin to avoid float jitter
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= patience:
                pbar.close()
                print(f"[Early stop] {title_prefix} — Fold {fold} @ epoch {epoch} (best val={best_val:.6f})")
                break

    # Restore best weights (by val loss) and produce final test preds/targets
    if best_state is not None:
        model.load_state_dict(best_state)

    val_loss, sp, neg_acc, preds, targets = test_epoch_single(model, test_emb, test_attn, test_labels, batch_size, l1_lambda, l2_lambda, loss, device)

    # per-fold plots (include epoch 0)
    plot_training_curves(history, title_prefix, outdir, fold)

    return history, preds, targets

# Experiments: test model performance on different datasets

In [11]:
# -----------------------
# Config
# -----------------------
DATASETS = [
    "Rike_high_confidence",
    "Rike_low_confidence",
    "Simon_ML",
    "Simon_ML+Michal_neg",
    "Max_lrDMS",
]
MODEL_TYPES = ["SeqRR", "EpiNN", "EpiATT", "SeqDT"] # choose among "SeqRR", "SeqDT", "EpiNN", "EpiATT"

NUM_FOLDS = 5
MAX_EPOCHS = 500
PATIENCE = 10
BATCH_SIZE = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULTS_ROOT = "/content/drive/MyDrive/Xai_ired/results"
OUTDIR = os.path.join(RESULTS_ROOT, "benchmark_5fold")
os.makedirs(OUTDIR, exist_ok=True)

# Model hyperparameters
HP = {
    "SeqRR": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
    "EpiNN": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
    "EpiATT": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
    "SeqDT": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
}

# -----------------------
# Metrics
# -----------------------
def spearman_pos_only(preds_np: np.ndarray, targets_np: np.ndarray) -> float:
    pos_mask = targets_np > CENSOR_THRESH
    if pos_mask.sum() < 2:
        return float("nan")
    rho, _ = spearmanr(preds_np[pos_mask], targets_np[pos_mask])
    return float(rho)

def precision_at_zero(preds_np: np.ndarray, targets_np: np.ndarray) -> float:
    y_true = (targets_np > 0.0).astype(np.int32)
    y_pred = (preds_np > 0.0).astype(np.int32)
    return float(precision_score(y_true, y_pred, zero_division=0))

# Percentage of top N better than WT
def precision_top_n(N: int, preds_np: np.ndarray, targets_np: np.ndarray) -> float:
    y_true = (targets_np > 0.0).astype(np.int32)
    y_pred = np.zeros(len(preds_np), dtype=np.int32)
    top_idx = np.argpartition(preds_np, -N)[-N:]  # indices of top N (unordered)
    y_pred[top_idx] = 1
    return float(precision_score(y_true, y_pred, zero_division=0))

def precision_top_n_simon(N: int, preds_np: np.ndarray, targets_np: np.ndarray) -> float:
    preds_np = np.asarray(preds_np).reshape(-1)
    targets_np = np.asarray(targets_np).reshape(-1)
    n = len(preds_np)
    top_idx = np.argpartition(preds_np, -N)[-N:]  # top N indices (unordered)

    hit = (preds_np[top_idx] > 0.0) & (targets_np[top_idx] > 0.0)
    return float(hit.mean())


# -----------------------
# One CV run for a (dataset, model)
# -----------------------
def run_cv(dataset: str, model_type: str) -> dict:
    data_dir = f"/content/drive/MyDrive/Xai_ired/Data/{dataset}"

    hp = HP[model_type]

    # Load embeddings (mmap helps RAM; slicing later copies only what’s needed)
    if model_type == "SeqRR":
        AAindex = np.load(os.path.join(data_dir, "AAindex_emb.npy"), mmap_mode="r")
        PLL = np.load(os.path.join(data_dir, "PLL.npy"), mmap_mode="r")
        ESM_emb, Attn_prior = None, None
    elif model_type == "EpiNN":
        AAindex = np.load(os.path.join(data_dir, "AAindex_emb_masked.npy"), mmap_mode="r")
        PLL = np.load(os.path.join(data_dir, "PLL.npy"), mmap_mode="r")
        L = AAindex.shape[1]-2
        D = AAindex.shape[2]
        AAindex = AAindex.reshape(AAindex.shape[0], -1)[:, 19:-19]
        ESM_emb, Attn_prior = None, None
    elif model_type == "EpiATT":
        ESM_emb = np.load(os.path.join(data_dir, "ESM_emb_masked.npy"), mmap_mode="r")
        L = ESM_emb.shape[1]
        D = ESM_emb.shape[2]
        Attn_prior = np.load(os.path.join(data_dir, "ESM_att.npy"), mmap_mode="r")
        AAindex= None
    else:
        ESM_emb = np.load(os.path.join(data_dir, "ESM_emb_masked.npy"), mmap_mode="r")
        L = ESM_emb.shape[1]
        D = ESM_emb.shape[2]
        AAindex, Attn_prior = None, None

    all_preds, all_targets = [], []
    best_epochs = []

    for fold in range(NUM_FOLDS):
        # indices + labels
        train_idx = pickle.load(open(os.path.join(data_dir, f"train/indices_{fold}.pkl"), "rb"))
        test_idx  = pickle.load(open(os.path.join(data_dir, f"test/indices_{fold}.pkl"), "rb"))

        train_labels = np.array(pickle.load(open(os.path.join(data_dir, f"train/labels_{fold}.pkl"), "rb")), dtype=np.float32)
        test_labels  = np.array(pickle.load(open(os.path.join(data_dir, f"test/labels_{fold}.pkl"), "rb")), dtype=np.float32)

        # slice embeddings
        if model_type == "SeqRR" or model_type == "EpiNN":
            train_emb = np.concatenate((AAindex[np.array(train_idx)], PLL[np.array(train_idx)]), axis=1)
            test_emb  = np.concatenate((AAindex[np.array(test_idx)],  PLL[np.array(test_idx)]),  axis=1)
            train_attn, test_attn = None, None
        elif model_type == "EpiATT":
            train_emb = ESM_emb[np.array(train_idx)]
            test_emb  = ESM_emb[np.array(test_idx)]
            train_attn = Attn_prior[np.array(train_idx)]
            test_attn  = Attn_prior[np.array(test_idx)]
        else:
            train_emb = ESM_emb[np.array(train_idx)]
            test_emb  = ESM_emb[np.array(test_idx)]
            train_attn, test_attn = None, None
        if model_type == "SeqRR":
            embedding_dim = train_emb.shape[1]
            loss_type = 'MSE'
            model = SeqRR(embedding_dim, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        elif model_type == 'EpiNN':
            loss_type = 'Huber'
            model = EpiNN(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        elif model_type == 'EpiATT':
            loss_type = 'Huber'
            model = EpiATT(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        else:
            loss_type = 'Huber'
            model = SeqDT(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)

        # train one fold (your existing tqdm progress happens inside)
        fold_outdir = os.path.join(OUTDIR, dataset, model_type, "per_fold")
        os.makedirs(fold_outdir, exist_ok=True)

        history, preds_t, targets_t = train_on_dataset(
            model=model,
            train_emb=train_emb, train_attn=train_attn, train_labels=train_labels,
            test_emb=test_emb,   test_attn=test_attn,   test_labels=test_labels,
            n_epoch=MAX_EPOCHS, batch_size=BATCH_SIZE, loss=loss_type, device=DEVICE,
            title_prefix=f"{model_type} | {dataset}",
            outdir=fold_outdir, fold=fold,
            lr=hp["lr"], weight_decay=hp["weight_decay"], dropout_rate=hp["dropout_rate"],
            l1_lambda=hp["l1_lambda"], l2_lambda=hp["l2_lambda"],
            patience=PATIENCE
        )

        # epoch index where val_loss is minimum (history includes epoch 0 baseline)
        best_epoch = int(np.argmin(np.asarray(history["val_loss"], dtype=float)))
        best_epochs.append(best_epoch)

        all_preds.append(preds_t.detach().cpu())
        all_targets.append(targets_t.detach().cpu())

        # free fold-sliced arrays ASAP to prevent data overload
        del train_emb, test_emb, train_attn, test_attn, model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # combine test across folds
    preds_np = torch.cat(all_preds).numpy()
    targets_np = torch.cat(all_targets).numpy()
    out_dir = os.path.join(OUTDIR, dataset, model_type)
    np.save(os.path.join(out_dir, "preds.npy"), preds_np)
    np.save(os.path.join(out_dir, "targets.npy"), targets_np)

    sp = spearman_pos_only(preds_np, targets_np)
    prec = precision_at_zero(preds_np, targets_np)
    top_prec = [precision_top_n(N, preds_np, targets_np) for N in [10, 20, 30, 40, 50]]
    top_prec_simon = [precision_top_n_simon(N, preds_np, targets_np) for N in [10, 20, 30, 40, 50]]
    avg_best_epoch = float(np.mean(best_epochs))

    # return summary
    return {
        "dataset": dataset,
        "model": model_type,
        "spearman_pos": sp,
        "precision_at_0": prec,
        "top_n_precision": top_prec,
        "top_n_precision_with_drop": top_prec_simon,
        "avg_best_epoch": avg_best_epoch,
        "best_epochs_per_fold": best_epochs,
        "n_test_total": int(len(targets_np)),
    }

# -----------------------
# Run all benchmarks
# -----------------------
all_rows = []
timestamp = time.strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(OUTDIR, f"cv_benchmark_{timestamp}.csv")
json_path = os.path.join(OUTDIR, f"cv_benchmark_{timestamp}.json")

outer = tqdm([(d, m) for d in DATASETS for m in MODEL_TYPES], desc="Benchmark runs")
for dataset, model_type in outer:
    res = run_cv(dataset, model_type)
    res['top_n_precision'] = ",".join(f"{x:.4f}" for x in res['top_n_precision'])
    res['top_n_precision_with_drop'] = ",".join(f"{x:.4f}" for x in res['top_n_precision_with_drop'])
    all_rows.append(res)

    print(f"\n[{dataset} | {model_type}] "
          f"Spearman(pos)={res['spearman_pos']:.4f} | "
          f"Precision@0={res['precision_at_0']:.4f} | "
          f"Top-N precision={res['top_n_precision']} | "
          f"Top-N precision with drop={res['top_n_precision_with_drop']} | "
          f"Avg best-epoch={res['avg_best_epoch']:.2f} | "
          f"Ntest={res['n_test_total']}")

    # incremental save after each CV (so you don’t lose progress)
    pd.DataFrame(all_rows).to_csv(csv_path, index=False)
    with open(json_path, "w") as f:
        json.dump(all_rows, f, indent=2)

print(f"\nSaved CSV:  {csv_path}")
print(f"Saved JSON: {json_path}")

Benchmark runs:   0%|          | 0/20 [00:00<?, ?it/s]

SeqRR | Rike_high_confidence — Fold 0:   0%|          | 0/500 [00:00<?, ?it/s]

[Early stop] SeqRR | Rike_high_confidence — Fold 0 @ epoch 60 (best val=0.926733)


SeqRR | Rike_high_confidence — Fold 1:   0%|          | 0/500 [00:00<?, ?it/s]

[Early stop] SeqRR | Rike_high_confidence — Fold 1 @ epoch 224 (best val=0.894342)


...

[Early stop] SeqDT | Max_lrDMS — Fold 2 @ epoch 108 (best val=1.286488)


SeqDT | Max_lrDMS — Fold 3:   0%|          | 0/500 [00:00<?, ?it/s]

[Early stop] SeqDT | Max_lrDMS — Fold 3 @ epoch 116 (best val=1.278118)


SeqDT | Max_lrDMS — Fold 4:   0%|          | 0/500 [00:00<?, ?it/s]

[Early stop] SeqDT | Max_lrDMS — Fold 4 @ epoch 56 (best val=1.284451)

[Max_lrDMS | SeqDT] Spearman(pos)=0.3187 | Precision@0=0.0000 | Top-N precision=0.3000,0.3500,0.4333,0.3750,0.3400 | Top-N precision with drop=0.0000,0.0000,0.0000,0.0000,0.0000 | Avg best-epoch=62.40 | Ntest=8589

Saved CSV:  /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/cv_benchmark_20260127_230456.csv
Saved JSON: /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/cv_benchmark_20260127_230456.json


# Experiments: retrain model and test on plate readouts

In [11]:
import glob
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

# -----------------------
# Settings
# -----------------------
DATASETS = [
    "Rike_high_confidence",
    "Rike_low_confidence",
    "Simon_ML",
    "Simon_ML+Michal_neg",
    "Max_lrDMS",
]
MODEL_TYPES = ["SeqRR", "EpiNN", "EpiATT", "SeqDT"]   # choose among "SeqRR", "SeqDT", "EpiNN", "EpiATT"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_ROOT = "/content/drive/MyDrive/Xai_ired/results"
BENCH_DIR = os.path.join(RESULTS_ROOT, "benchmark_5fold")  # where your cv_benchmark_*.csv lives
OUTDIR = os.path.join(BENCH_DIR, "full_train_and_plate_eval")
os.makedirs(OUTDIR, exist_ok=True)

BATCH_SIZE_TRAIN = 64
BATCH_SIZE_EVAL  = 128


HP = {
    "SeqRR": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
    "SeqDT": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
    "EpiNN": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
    "EpiATT": dict(lr=3e-4, weight_decay=1e-3, dropout_rate=0.0, l1_lambda=5e-3, l2_lambda=0.0),
}

torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# Helpers: find latest benchmark CSV
# -----------------------
def find_latest_benchmark_csv(bench_dir: str) -> str:
    cands = sorted(glob.glob(os.path.join(bench_dir, "cv_benchmark_*.csv")))
    if not cands:
        raise FileNotFoundError(f"No cv_benchmark_*.csv found under: {bench_dir}")
    return cands[-1]

bench_csv = find_latest_benchmark_csv(BENCH_DIR)
bench_df = pd.read_csv(bench_csv)
print("Using benchmark file:", bench_csv)

# build dict of avg epochs
avg_epoch_map = {}
for _, r in bench_df.iterrows():
    avg_epoch_map[(r["dataset"], r["model"])] = float(r["avg_best_epoch"])

def get_full_epochs(dataset: str, model_type: str) -> int:
    e = avg_epoch_map.get((dataset, model_type), None)
    if e is None or (isinstance(e, float) and np.isnan(e)):
        # fallback if missing
        return 50
    # history includes epoch 0 baseline -> for full training, use at least 1 epoch
    return max(1, int(math.ceil(e)))

# -----------------------
# Batch loader by ORIGINAL INDICES (no huge slicing)
# -----------------------
def load_batch_by_original_indices(emb_mmap, attn_mmap, labels_arr, orig_indices, start, end, device):
    # orig_indices: array of original row indices into embedding files
    batch_idx = orig_indices[start:end]
    x = torch.from_numpy(np.asarray(emb_mmap[batch_idx], dtype=np.float32)).to(device)
    if attn_mmap is not None:
        a = torch.from_numpy(np.asarray(attn_mmap[batch_idx], dtype=np.float32)).to(device)
    else:
        a = None
    y = torch.from_numpy(np.asarray(labels_arr[start:end], dtype=np.float32)).to(device)
    return x, a, y

def train_one_epoch_indexed(model, optimizer, emb_mmap, attn_mmap, labels_arr, orig_indices, batch_size, l1_lambda, l2_lambda, loss):
    model.train()
    N = len(labels_arr)
    running, total = 0.0, 0
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        x, a, y = load_batch_by_original_indices(emb_mmap, attn_mmap, labels_arr, orig_indices, start, end, DEVICE)

        if a is not None:
            preds = model(x, a).squeeze(-1)
        else:
            preds = model(x).squeeze(-1)

        if loss == "MSE":
            loss = censored_mse_loss(model, preds, y, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        else:
            loss = censored_huber_plus_rank_loss(model, preds, y, l1_lambda=l1_lambda, l2_lambda=l2_lambda)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bs = (end - start)
        running += float(loss.item()) * bs
        total += bs

    return running / max(1, total)

@torch.no_grad()
def predict_from_arrays(model, emb_arr, attn_arr=None, batch_size=128):
    model.eval()
    N = emb_arr.shape[0]
    preds = []
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        x = torch.from_numpy(np.asarray(emb_arr[start:end], dtype=np.float32)).to(DEVICE)
        if attn_arr is not None:
            a = torch.from_numpy(np.asarray(attn_arr[start:end], dtype=np.float32)).to(DEVICE)
            p = model(x, a)
        else:
            p = model(x)
        preds.append(p.squeeze(-1).detach().cpu().numpy())
    return np.concatenate(preds, axis=0)

# -----------------------
# Full-train set: concat test folds
# -----------------------
def build_full_train_from_test_folds(dataset: str):
    base = f"/content/drive/MyDrive/Xai_ired/Data/{dataset}"
    all_idx, all_lab = [], []
    for fold in range(5):
        idx = pickle.load(open(os.path.join(base, f"test/indices_{fold}.pkl"), "rb"))
        lab = pickle.load(open(os.path.join(base, f"test/labels_{fold}.pkl"), "rb"))
        all_idx.append(np.asarray(idx, dtype=np.int64))
        all_lab.append(np.asarray(lab, dtype=np.float32))
    orig_indices = np.concatenate(all_idx, axis=0)
    labels = np.concatenate(all_lab, axis=0)

    # sanity
    if len(orig_indices) != len(labels):
        raise ValueError(f"[{dataset}] indices/labels length mismatch: {len(orig_indices)} vs {len(labels)}")

    return orig_indices, labels

# -----------------------
# Train full model for one dataset+model
# -----------------------
def train_full_model(dataset: str, model_type: str, epochs: int):
    hp = HP[model_type]
    base = f"/content/drive/MyDrive/Xai_ired/Data/{dataset}"

    orig_indices, labels = build_full_train_from_test_folds(dataset)


    if model_type == "SeqRR":
        AA = np.load(os.path.join(base, "AAindex_emb.npy"))
        PL = np.load(os.path.join(base, "PLL.npy"))
        EMB = np.concatenate([AA, PL], axis=1)
        ATT = None
        embedding_dim = EMB.shape[1]
        loss_type = "MSE"
        model = SeqRR(embedding_dim, dropout_rate=hp["dropout_rate"]).to(DEVICE)

    elif model_type == "EpiNN":
        AA = np.load(os.path.join(base, "AAindex_emb_masked.npy"))
        PL = np.load(os.path.join(base, "PLL.npy"))
        L = AA.shape[1]-2
        D = AA.shape[2]
        AA = AA.reshape(AA.shape[0], -1)[:, 19:-19]
        EMB = np.concatenate([AA, PL], axis=1)
        ATT = None
        loss_type = "Huber"
        model = EpiNN(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)

    elif model_type == "EpiATT":
        ESM = np.load(os.path.join(base, "ESM_emb_masked.npy"))
        L = ESM.shape[1]
        D = ESM.shape[2]
        EMB = ESM
        ATT = np.load(os.path.join(base, "ESM_att.npy"))
        loss_type = "Huber"
        model = EpiATT(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)

    else:
        ESM = np.load(os.path.join(base, "ESM_emb_masked.npy"))
        L = ESM.shape[1]
        D = ESM.shape[2]
        EMB = ESM
        ATT = None
        loss_type = "Huber"
        model = SeqDT(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    pbar = tqdm(range(1, epochs + 1), desc=f"Full train | {dataset} | {model_type}")
    for ep in pbar:
        train_loss = train_one_epoch_indexed(
            model, optimizer, EMB, ATT, labels, orig_indices,
            batch_size=BATCH_SIZE_TRAIN,
            l1_lambda=hp["l1_lambda"], l2_lambda=hp["l2_lambda"], loss=loss_type
        )
        pbar.set_postfix({"train_loss": f"{train_loss:.4f}"})

    # save
    save_dir = os.path.join(OUTDIR, "full_models")
    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"{dataset}__{model_type}__full.pt")
    torch.save(model.state_dict(), ckpt_path)

    # also save meta
    meta_path = os.path.join(save_dir, f"{dataset}__{model_type}__full_meta.json")
    with open(meta_path, "w") as f:
        json.dump({"dataset": dataset, "model": model_type, "epochs": epochs, "hp": hp, "timestamp": time.ctime()}, f, indent=2)

    # cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return ckpt_path

# -----------------------
# Evaluate on plate dataset
# -----------------------
def eval_on_plate(dataset: str, model_type: str, ckpt_path: str):
    hp = HP[model_type]
    base = f"/content/drive/MyDrive/Xai_ired/Data/{dataset}"

    # labels
    plate_csv = os.path.join(base, "plate_dataset.csv")
    df_plate = pd.read_csv(plate_csv)
    y = pd.to_numeric(df_plate["1/s"], errors="coerce").to_numpy(dtype=np.float32)
    y = np.log(y)

    # embeddings aligned to csv order
    eval_dir = os.path.join(base, "evaluation")

    if model_type == "SeqRR":
        AA = np.load(os.path.join(eval_dir, "AAindex_emb.npy"), mmap_mode="r")
        PL = np.load(os.path.join(eval_dir, "PLL.npy"), mmap_mode="r")
        EMB = np.concatenate([AA, PL], axis=1)
        ATT = None
        embedding_dim = EMB.shape[1]
        model = SeqRR(embedding_dim, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    elif model_type == "EpiNN":
        AA = np.load(os.path.join(eval_dir, "AAindex_emb_masked.npy"), mmap_mode="r")
        L = AA.shape[1]-2
        D = AA.shape[2]
        AA = AA.reshape(AA.shape[0], -1)[:, 19:-19]
        PL = np.load(os.path.join(eval_dir, "PLL.npy"), mmap_mode="r")
        EMB = np.concatenate([AA, PL], axis=1)
        ATT = None
        model = EpiNN(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    elif model_type == "EpiATT":
        ESM = np.load(os.path.join(eval_dir, "ESM_emb_masked.npy"), mmap_mode="r")
        L = ESM.shape[1]
        D = ESM.shape[2]
        EMB = ESM
        ATT = np.load(os.path.join(eval_dir, "ESM_att.npy"), mmap_mode="r")
        model = EpiATT(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    else:
        ESM = np.load(os.path.join(eval_dir, "ESM_emb_masked.npy"), mmap_mode="r")
        L = ESM.shape[1]
        D = ESM.shape[2]
        EMB = ESM
        ATT = None
        model = SeqDT(L, D, dropout_rate=hp["dropout_rate"]).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    # predict all then mask
    N = min(len(y), EMB.shape[0])
    y = y[:N]
    mask = np.isfinite(y)
    y_pred_all = predict_from_arrays(model, EMB[:N], ATT, batch_size=BATCH_SIZE_EVAL)
    y_true = y[mask]
    y_pred = y_pred_all[mask]


    # metrics
    if len(y_true) < 2:
        pr = float("nan")
        r2 = float("nan")
        sp = float("nan")
        top10_jaccard = float("nan")
    else:
        pr = float(pearsonr(y_true, y_pred)[0])
        sp = float(spearmanr(y_true, y_pred).correlation)

        # --- NEW: linear calibration y_true ≈ a*y_pred + b ---
        lr_cal = LinearRegression()
        lr_cal.fit(y_pred.reshape(-1, 1), y_true)
        y_pred_cal = lr_cal.predict(y_pred.reshape(-1, 1))
        r2 = float(r2_score(y_true, y_pred_cal))

        valid_idx = np.where(mask)[0]   # indices in 0..N-1 that have finite y
        k = min(10, len(y_true))
        # top-k by prediction and by truth (descending)
        top_pred_idx = set(valid_idx[np.argsort(y_pred)[-k:]].tolist())
        top_true_idx = set(valid_idx[np.argsort(y_true)[-k:]].tolist())
        inter = len(top_pred_idx & top_true_idx)
        union = len(top_pred_idx | top_true_idx)
        top10_jaccard = inter / union if union > 0 else float("nan")


    # attach predictions and save
    out_pred_csv = os.path.join(OUTDIR, "plate_preds")
    os.makedirs(out_pred_csv, exist_ok=True)
    df_out = df_plate.iloc[:N].copy()
    df_out["pred_fitness"] = np.nan
    df_out.loc[mask, "pred_fitness"] = y_pred
    df_out["pred_fitness_cal"] = np.nan
    df_out.loc[mask, "pred_fitness_cal"] = y_pred_cal
    df_out.to_csv(os.path.join(out_pred_csv, f"{dataset}__{model_type}__plate_preds.csv"), index=False)

    return pr, r2, sp, top10_jaccard, int(np.sum(mask)), int(N)

# -----------------------
# Run all: full-train + plate eval
# -----------------------
rows = []
for dataset in DATASETS:
    for model_type in MODEL_TYPES:
        epochs = get_full_epochs(dataset, model_type)
        print(f"\n=== {dataset} | {model_type} | full-train epochs={epochs} ===")

        ckpt_path = train_full_model(dataset, model_type, epochs=epochs)

        pr, r2, sp, jac10, n_valid, n_total = eval_on_plate(dataset, model_type, ckpt_path)

        print(f"[Plate eval] Pearson r={pr:.4f} | Spearman={sp:.4f} | R^2={r2:.4f} | Top10-Jaccard={jac10:.4f} | valid={n_valid}/{n_total}")
        rows.append({
            "dataset": dataset,
            "model": model_type,
            "full_train_epochs": epochs,
            "ckpt_path": ckpt_path,
            "plate_pearson_r": pr,
            "plate_spearman": sp,
            "plate_r2": r2,
            "plate_top10_jaccard": jac10,
            "plate_valid_n": n_valid,
            "plate_total_n": n_total,
        })

        # keep saving progress
        pd.DataFrame(rows).to_csv(os.path.join(OUTDIR, f"plate_eval_metrics_{'_'.join(MODEL_TYPES)}.csv"), index=False)
        with open(os.path.join(OUTDIR, f"plate_eval_metrics_{'_'.join(MODEL_TYPES)}.json"), "w") as f:
            json.dump(rows, f, indent=2)

print("\nSaved metrics to:")
print(" -", os.path.join(OUTDIR, f"plate_eval_metrics_{'_'.join(MODEL_TYPES)}.csv"))
print(" -", os.path.join(OUTDIR, f"plate_eval_metrics_{'_'.join(MODEL_TYPES)}.json"))

Using benchmark file: /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/cv_benchmark_20260127_230456.csv

=== Rike_high_confidence | SeqRR | full-train epochs=152 ===


Full train | Rike_high_confidence | SeqRR:   0%|          | 0/152 [00:00<?, ?it/s]

[Plate eval] Pearson r=0.5651 | Spearman=0.5319 | R^2=0.3193 | Top10-Jaccard=0.3333 | valid=54/55

=== Rike_high_confidence | EpiNN | full-train epochs=90 ===


/tmp/ipython-input-2233121139.py:230: RuntimeWarning: invalid value encountered in log
  y = np.log(y)


Full train | Rike_high_confidence | EpiNN:   0%|          | 0/90 [00:00<?, ?it/s]

...

/tmp/ipython-input-2233121139.py:230: RuntimeWarning: invalid value encountered in log
  y = np.log(y)
/tmp/ipython-input-2233121139.py:289: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  pr = float(pearsonr(y_true, y_pred)[0])


[Plate eval] Pearson r=-0.0884 | Spearman=-0.0563 | R^2=0.0050 | Top10-Jaccard=0.0526 | valid=51/52

=== Max_lrDMS | SeqDT | full-train epochs=63 ===


Full train | Max_lrDMS | SeqDT:   0%|          | 0/63 [00:00<?, ?it/s]

[Plate eval] Pearson r=0.1887 | Spearman=0.1750 | R^2=0.0356 | Top10-Jaccard=0.0526 | valid=51/52

Saved metrics to:
 - /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/full_train_and_plate_eval/plate_eval_metrics_SeqRR_EpiNN_EpiATT_SeqDT.csv
 - /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/full_train_and_plate_eval/plate_eval_metrics_SeqRR_EpiNN_EpiATT_SeqDT.json


/tmp/ipython-input-2233121139.py:230: RuntimeWarning: invalid value encountered in log
  y = np.log(y)


# Plot model performance on the CV and eval experiments

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


MODEL_TYPES = ["SeqRR", "EpiNN", "EpiATT", "SeqDT"] # SeqRR or EpiNN or EpiATT or SeqDT
DATASETS = [
    "Rike_high_confidence",
    "Rike_low_confidence",
    "Simon_ML",
    "Simon_ML+Michal_neg",
    "Max_lrDMS",
]
RESULTS_ROOT = "/content/drive/MyDrive/Xai_ired/results"
OUTDIR = os.path.join(RESULTS_ROOT, "benchmark_5fold")
BENCH_DIR = os.path.join(OUTDIR, "full_train_and_plate_eval")


# -----------------------
# Helper: find latest CV benchmark CSV
# -----------------------
def find_latest_benchmark_csv(outdir: str) -> str:
    cands = sorted(glob.glob(os.path.join(outdir, "cv_benchmark*.csv")))
    if not cands:
        raise FileNotFoundError(f"No cv_benchmark*_patchedTopN.csv found under: {outdir}")
    return cands[-1]

# -----------------------
# Helper: parse "top_n_precision" column (string like "0.1234,0.2345,...")
# -----------------------
def parse_top10_precision(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (list, tuple, np.ndarray)):
        return float(val[0]) if len(val) else np.nan
    s = str(val).strip()
    if not s:
        return np.nan
    try:
        return float(s.split(",")[0])
    except Exception:
        return np.nan

# -----------------------
# Build plot for one model
# -----------------------
def plot_model_bars(model_type: str, datasets, bench_dir: str, outdir: str):
    # Load CV metrics (latest file)
    cv_csv = find_latest_benchmark_csv(outdir)
    cv_df = pd.read_csv(cv_csv)

    # Load plate metrics
    plate_csv = os.path.join(bench_dir, f"plate_eval_metrics_{'_'.join(MODEL_TYPES)}.csv")
    if not os.path.exists(plate_csv):
        raise FileNotFoundError(f"plate_eval_metrics.csv not found at: {plate_csv}")
    plate_df = pd.read_csv(plate_csv)

    metric_names = [
        "CV Spearman",
        "CV Precision",
        "CV Top10 precision w/t drop",
        "Plate Pearson r",
        "Plate R²",
        "Plate Top10 Jaccard",
    ]

    # Assemble a (num_datasets, 6) matrix
    M = np.full((len(datasets), len(metric_names)), np.nan, dtype=float)

    for i, ds in enumerate(datasets):
        # --- CV row ---
        cv_row = cv_df[(cv_df["dataset"] == ds) & (cv_df["model"] == model_type)]
        if len(cv_row) > 0:
            r = cv_row.iloc[-1]
            M[i, 0] = float(r.get("spearman_pos", np.nan))
            M[i, 1] = float(r.get("precision_at_0", np.nan))
            M[i, 2] = parse_top10_precision(r.get("top_n_precision_with_drop", np.nan))

        # --- Plate row ---
        pl_row = plate_df[(plate_df["dataset"] == ds) & (plate_df["model"] == model_type)]
        if len(pl_row) > 0:
            r = pl_row.iloc[-1]
            M[i, 3] = float(r.get("plate_pearson_r", np.nan))
            M[i, 4] = float(r.get("plate_r2", np.nan))
            M[i, 5] = float(r.get("plate_top10_jaccard", np.nan))

    # -----------------------
    # Plot grouped bars
    # -----------------------
    x = np.arange(len(datasets))
    n_metrics = M.shape[1]
    width = min(0.12, 0.85 / n_metrics)  # keep groups from overflowing
    offsets = (np.arange(n_metrics) - (n_metrics - 1) / 2.0) * width

    plt.figure(figsize=(16, 6))
    for j in range(n_metrics):
        plt.bar(x + offsets[j], M[:, j], width=width, label=metric_names[j])

    plt.xticks(x, datasets, rotation=20, ha="right")
    plt.xlabel("Dataset")
    plt.ylabel("Metric value")
    plt.title(f"{model_type} — CV + Plate metrics by dataset")
    plt.legend(ncol=3, fontsize=9)
    plt.tight_layout()

    out_path = os.path.join(outdir, f"{model_type}_barplot_cv_plate_metrics.png")
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"Saved: {out_path}")

# -----------------------
# Run plotting for both models
# -----------------------
for model_type in MODEL_TYPES:
    plot_model_bars(
        model_type=model_type,
        datasets=DATASETS,
        bench_dir=BENCH_DIR,
        outdir=OUTDIR
    )

Saved: /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/SeqRR_barplot_cv_plate_metrics.png
Saved: /content/drive/MyDrive/Xai_ired/results/benchmark_5fold/EpiNN_barplot_cv_plate_metrics.png


# Investigate the weird top N precision performance
## reformulate top N 10 precision

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def precision_top_n_simon(N: int, preds_np: np.ndarray, targets_np: np.ndarray) -> float:
    preds_np = np.asarray(preds_np).reshape(-1)
    targets_np = np.asarray(targets_np).reshape(-1)
    n = len(preds_np)
    top_idx = np.argpartition(preds_np, -N)[-N:]  # top N indices (unordered)

    hit = (preds_np[top_idx] > 0.0) & (targets_np[top_idx] > 0.0)
    return float(hit.mean())



def find_latest_benchmark_csv(outdir: str) -> str:
    cands = sorted(glob.glob(os.path.join(outdir, "cv_benchmark_*.csv")))
    if not cands:
        raise FileNotFoundError(f"No cv_benchmark_*.csv found under: {outdir}")
    return cands[-1]

RESULTS_ROOT = "/content/drive/MyDrive/Xai_ired/results"
OUTDIR = os.path.join(RESULTS_ROOT, "benchmark_5fold")
DATASETS = [
    "Rike_high_confidence",
    "Rike_low_confidence",
    "Simon_ML",
    "Simon_ML+Michal_neg",
    "Max_lrDMS",
]
MODEL_TYPES = ["SeqRR", "SeqDT"]

csv_path = find_latest_benchmark_csv(OUTDIR)
df_bench = pd.read_csv(csv_path)

# Recompute top_n_precision for each row using saved preds/targets
new_topn_strs = []
for _, row in df_bench.iterrows():
    dataset = row["dataset"]
    model_type = row["model"]

    out_dir = os.path.join(OUTDIR, dataset, model_type)
    preds_path = os.path.join(out_dir, "preds.npy")
    targets_path = os.path.join(out_dir, "targets.npy")

    if not (os.path.exists(preds_path) and os.path.exists(targets_path)):
        # keep existing if missing
        new_topn_strs.append(row.get("top_n_precision", ""))
        continue

    preds_np = np.load(preds_path)
    targets_np = np.load(targets_path)

    # compute Top-10/20/30/40/50 with new logic
    vals = [precision_top_n_simon(N, preds_np, targets_np) for N in [10, 20, 30, 40, 50]]
    topn_str = ",".join(f"{v:.4f}" for v in vals)
    new_topn_strs.append(topn_str)
    print(f"[{os.path.basename(csv_path)}] {dataset} | {model_type} | TopN={topn_str}")

df_bench["top_n_precision"] = new_topn_strs

# Save a patched copy (safer than overwriting)
patched_path = csv_path.replace(".csv", "_patchedTopN.csv")
df_bench.to_csv(patched_path, index=False)
print(f"Patched top_n_precision saved to: {patched_path}")

for model_type in MODEL_TYPES:
    for dataset in DATASETS:
        print(f"\n=== {dataset} | {model_type} ===")
        out_dir = os.path.join(OUTDIR, dataset, model_type)
        preds_np = np.load(os.path.join(out_dir, "preds.npy"))
        targets_np = np.load(os.path.join(out_dir, "targets.npy"))
        idx = np.argsort(preds_np)[::-1]
        preds_sorted = preds_np[idx]
        targets_sorted = targets_np[idx]
        print(preds_sorted[:30])
        print(targets_sorted[:30])

[cv_benchmark_20260121_215016_patchedTopN.csv] Rike_high_confidence | SeqRR | TopN=1.0000,1.0000,1.0000,1.0000,1.0000
[cv_benchmark_20260121_215016_patchedTopN.csv] Rike_high_confidence | EpiNN | TopN=1.0000,1.0000,1.0000,1.0000,1.0000
[cv_benchmark_20260121_215016_patchedTopN.csv] Rike_low_confidence | SeqRR | TopN=1.0000,1.0000,1.0000,1.0000,1.0000
[cv_benchmark_20260121_215016_patchedTopN.csv] Rike_low_confidence | EpiNN | TopN=1.0000,1.0000,1.0000,1.0000,1.0000
[cv_benchmark_20260121_215016_patchedTopN.csv] Simon_ML | SeqRR | TopN=0.7000,0.7500,0.8000,0.8500,0.8800
[cv_benchmark_20260121_215016_patchedTopN.csv] Simon_ML | EpiNN | TopN=0.8000,0.8500,0.9000,0.9250,0.9400
[cv_benchmark_20260121_215016_patchedTopN.csv] Simon_ML+Michal_neg | SeqRR | TopN=0.9000,0.9500,0.9667,0.9500,0.9400
[cv_benchmark_20260121_215016_patchedTopN.csv] Simon_ML+Michal_neg | EpiNN | TopN=1.0000,0.9500,0.9333,0.9500,0.9600
[cv_benchmark_20260121_215016_patchedTopN.csv] Max_lrDMS | SeqRR | TopN=0.2000,0.100